In [4]:
import os
import json
from glob import glob

# Get all JSON files in the current folder
json_files = glob('*.json')

unique_dict = {}

for file in json_files:
    with open(file, 'r', encoding='utf-8') as f:
        try:
            data = json.load(f)
            # Use filename as key to ensure uniqueness
            unique_dict[file] = data
        except Exception as e:
            print(f'Error reading {file}: {e}')

print(f"Loaded {len(unique_dict)} files into unique_dict.")

Loaded 12 files into unique_dict.


In [ ]:
unique_dict

In [ ]:
# Extract all JSON filenames and add them to a dictionary
filenames_dict = {i: fname for i, fname in enumerate(json_files)}
filenames_dict

# Extract image file names from each JSON file and add them to a list
image_name = []
for file, content in unique_dict.items():
    # Try to extract image file names from common keys
    if isinstance(content, dict):
        # If the JSON is a dict, look for keys that might contain image names
        for k, v in content.items():
            if isinstance(v, str) and (v.endswith('.jpg') or v.endswith('.png')):
                image_name.append(v)
            elif isinstance(v, list):
                for item in v:
                    if isinstance(item, str) and (item.endswith('.jpg') or item.endswith('.png')):
                        image_name.append(item)
                    elif isinstance(item, dict):
                        for val in item.values():
                            if isinstance(val, str) and (val.endswith('.jpg') or val.endswith('.png')):
                                image_name.append(val)
    elif isinstance(content, list):
        # If the JSON is a list, iterate through items
        for item in content:
            if isinstance(item, dict):
                for val in item.values():
                    if isinstance(val, str) and (val.endswith('.jpg') or val.endswith('.png')):
                        image_name.append(val)
            elif isinstance(item, str) and (item.endswith('.jpg') or item.endswith('.png')):
                image_name.append(item)

image_name

In [9]:
len(set(image_name))


251

In [67]:
filenames_dict

{0: 'batch1_1-14.json',
 1: 'batch1_15-50.json',
 2: 'batch25_5.json',
 3: 'batch25_5_2.json',
 4: 'batch2_51-111.json',
 5: 'batch_final.json',
 6: 'labels_my-project-name_2025-06-03-11-56-45.json',
 7: 'segmentation_Sam27_5.json',
 8: 'spigole.json'}

In [65]:
len(image_name)

369

In [66]:
len(set(image_name))

251

In [69]:
# save the list to a text file
with open('image_file_names.txt', 'w') as f:
    for name in set(image_name):
        f.write(f"{name}\n")


In [51]:
image_name[0]

'1696503307photo_l1.jpg'

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="L0AQLsKjUqaH7giMGxMR")
project = rf.workspace("fishai-blwv5").project("freshness-dzd8j-dkujk")
version = project.version(2)
dataset = version.download("sam2")

In [10]:
rf = Roboflow(api_key="L0AQLsKjUqaH7giMGxMR")
project = rf.workspace("fishai-blwv5").project("fish-sam-segmentation") # new dataset
version = project.version(2)
dataset = version.download("sam2")

loading Roboflow workspace...
loading Roboflow project...
loading Roboflow project...
Exporting format sam2 in progress : 85.0%
Version export complete for sam2 format

Version export complete for sam2 format



Extracting Dataset Version Zip to Fish-SAM-Segmentation--2 in sam2:: 100%|██████████| 419/419 [00:00<00:00, 465.04it/s]
Extracting Dataset Version Zip to Fish-SAM-Segmentation--2 in sam2:: 100%|██████████| 419/419 [00:00<00:00, 465.04it/s]


In [33]:
from pathlib import Path

In [39]:
seg2_imgs = Path('Fish-SAM-Segmentation--2/train/').glob('*.jpg')

In [40]:
s2 = [x.name for x in seg2_imgs]

In [18]:
segm2 = list(glob('Fish-SAM-Segmentation--2/train/*.*'))

In [ ]:
segm2

In [22]:
image_name[0]

'1696503307photo_l1.jpg'

In [31]:
fname = "Fish-SAM-Segmentation--2\train\1696503307photo_l1_jpg.rf.22bb098f3ffeb3f920cba19b10c59a43.jpg"

In [32]:
image_name[0] in str(fname)

False

In [30]:
str(fname)

'1731402396_photo-1-_l4_jpg.rf.9c3d43f9edacf7c30859a4e0a5e69676.json'

In [29]:
for fname in segm2:
    if str(img) in str(fname):
        print(fname)

In [26]:
any(img in fname for fname in segm2)

False

In [43]:
s2[:5]

['1696503307photo_l1_jpg.rf.22bb098f3ffeb3f920cba19b10c59a43.jpg',
 '1696503342photo_l1_jpg.rf.424f2863a1e68a71ee97be6885c03cda.jpg',
 '1696503342photo_l2_jpg.rf.3edecfd070cdc5ac79931c3a6533c3ef.jpg',
 '1696503342photo_l3_jpg.rf.9c828483941c4c49611267421ea2ef9a.jpg',
 '1696503342photo_l4_jpg.rf.e9941b7be80e2d7a44248b7f5dc5ce3e.jpg']

In [56]:
# Load the merged data from segmantation_training_data.json
with open('segmantation_training_data.json', 'r', encoding='utf-8') as f:
    merged_data = json.load(f)

# Filter records where the image is in missing_images
filtered_data = []
for record in merged_data:
    # Try to find the image name in the record (common keys: 'image', 'file_name', etc.)
    img_name = None
    if isinstance(record, dict):
        for key in ['image', 'file_name', 'filename', 'img', 'img_name']:
            if key in record:
                img_name = record[key]
                break
    if img_name and img_name in missing_images:
        filtered_data.append(record)

# Save filtered data to missing_images_train.json
with open('missing_images_train.json', 'w', encoding='utf-8') as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(filtered_data)} records to missing_images_train.json.")

Saved 0 records to missing_images_train.json.


In [57]:
merged_data

[{'info': {'description': 'my-project-name'},
  'images': [{'id': 1,
    'width': 3472,
    'height': 4576,
    'file_name': '1696503307photo_l1.jpg'},
   {'id': 2,
    'width': 3472,
    'height': 4576,
    'file_name': '1696503359photo_l3.jpg'},
   {'id': 3,
    'width': 3472,
    'height': 4576,
    'file_name': '1696503582photo_l2.jpg'},
   {'id': 4,
    'width': 3472,
    'height': 4576,
    'file_name': '1696504577photo_l3.jpg'},
   {'id': 5,
    'width': 3472,
    'height': 4576,
    'file_name': '1696504594photo_l4.jpg'},
   {'id': 6,
    'width': 3472,
    'height': 4576,
    'file_name': '1696504611photo_l3.jpg'},
   {'id': 7,
    'width': 3472,
    'height': 4576,
    'file_name': '1696504611photo_l4.jpg'},
   {'id': 8,
    'width': 3472,
    'height': 4576,
    'file_name': '1696504628photo_l1.jpg'},
   {'id': 9,
    'width': 3472,
    'height': 4576,
    'file_name': '1696504628photo_l2.jpg'},
   {'id': 10,
    'width': 3472,
    'height': 4576,
    'file_name': '169650464

In [44]:
image_name[0]

'1696503307photo_l1.jpg'

In [45]:
s2 = [s2]

['1696503307photo_l1_jpg.rf.22bb098f3ffeb3f920cba19b10c59a43.jpg',
 '1696503342photo_l1_jpg.rf.424f2863a1e68a71ee97be6885c03cda.jpg',
 '1696503342photo_l2_jpg.rf.3edecfd070cdc5ac79931c3a6533c3ef.jpg',
 '1696503342photo_l3_jpg.rf.9c828483941c4c49611267421ea2ef9a.jpg',
 '1696503342photo_l4_jpg.rf.e9941b7be80e2d7a44248b7f5dc5ce3e.jpg']

In [ ]:
merged_data

In [55]:
missing_images

['1696504850photo_l2.jpg',
 '1731401664_photo (1)_l1.jpg',
 '1731401664_photo (1)_l3.jpg',
 '1731402040_photo (1)_l3.jpg',
 '1731402396_photo (1)_l4.jpg',
 '1696507792photo_l4.jpg',
 '1696503700photo_l4.jpg',
 '1696504714photo_l4.jpg',
 '1696504800photo_l1.jpg',
 '1696504884photo_l3.jpg',
 '1696504901photo_l3.jpg',
 '1696505020photo_l1.jpg',
 '1696505053photo_l1.jpg',
 '1696506672photo_l1.jpg',
 '1696506773photo_l3.jpg',
 '1696506773photo_l4.jpg',
 '1696506790photo_l3.jpg',
 '1696506807photo_l1.jpg',
 '1696506824photo_l1.jpg',
 '1696506859photo_l3.jpg',
 '1696506947photo_l1.jpg',
 '1696507082photo_l2.jpg',
 '1696507082photo_l4.jpg',
 '1696507538photo_l4.jpg',
 '1696507572photo_l1.jpg',
 '1696507792photo_l2.jpg',
 '1715784499_photo_l1.jpg',
 '1715784663_photo_l2.jpg',
 '1715784673_photo_l1.jpg',
 '1715785197_photo_l3.jpg',
 '1715785312_photo_l4.jpg',
 '1715785332_photo_l1.jpg',
 '1715785351_photo_l4.jpg',
 '1715786009_photo_l3.jpg',
 '1715786114_photo_l1.jpg',
 '1715786654_photo_l2.jpg'

In [61]:
# Inspect the structure of the first few records in merged_data to determine the correct key for image file names
for i, record in enumerate(merged_data[:5]):
    print(f'Record {i}:', record)

# Try to extract image file names using the correct key after inspection
image_file_names = []
possible_keys = ['image', 'file_name', 'filename', 'img', 'img_name']
for record in merged_data:
    img_name = None
    if isinstance(record, dict):
        # Try all possible keys, but also check nested structures if needed
        for key in possible_keys:
            if key in record:
                img_name = record[key]
                break
        # If not found, check for nested dicts or lists
        if not img_name:
            for v in record.values():
                if isinstance(v, dict):
                    for key in possible_keys:
                        if key in v:
                            img_name = v[key]
                            break
                elif isinstance(v, list):
                    for item in v:
                        if isinstance(item, dict):
                            for key in possible_keys:
                                if key in item:
                                    img_name = item[key]
                                    break
                        if img_name:
                            break
                if img_name:
                    break
    if img_name:
        image_file_names.append(img_name)

print(f"Extracted {len(image_file_names)} image file names.")

Record 0: {'info': {'description': 'my-project-name'}, 'images': [{'id': 1, 'width': 3472, 'height': 4576, 'file_name': '1696503307photo_l1.jpg'}, {'id': 2, 'width': 3472, 'height': 4576, 'file_name': '1696503359photo_l3.jpg'}, {'id': 3, 'width': 3472, 'height': 4576, 'file_name': '1696503582photo_l2.jpg'}, {'id': 4, 'width': 3472, 'height': 4576, 'file_name': '1696504577photo_l3.jpg'}, {'id': 5, 'width': 3472, 'height': 4576, 'file_name': '1696504594photo_l4.jpg'}, {'id': 6, 'width': 3472, 'height': 4576, 'file_name': '1696504611photo_l3.jpg'}, {'id': 7, 'width': 3472, 'height': 4576, 'file_name': '1696504611photo_l4.jpg'}, {'id': 8, 'width': 3472, 'height': 4576, 'file_name': '1696504628photo_l1.jpg'}, {'id': 9, 'width': 3472, 'height': 4576, 'file_name': '1696504628photo_l2.jpg'}, {'id': 10, 'width': 3472, 'height': 4576, 'file_name': '1696504645photo_l1.jpg'}, {'id': 11, 'width': 3472, 'height': 4576, 'file_name': '1696504645photo_l3.jpg'}, {'id': 12, 'width': 3472, 'height': 4576,

In [ ]:
merged_data

In [62]:
image_file_names

['1696503307photo_l1.jpg',
 '1696504679photo_l1.jpg',
 '1696504577photo_l3.jpg',
 '1715785351_photo_l2.jpg',
 '1696507065photo_l1.jpg',
 '1696503307photo_l1.jpg',
 '1696507031photo_l2.jpg',
 '1696503700photo_l4.jpg',
 '1696503307photo_l1.jpg']